## 1. Problem Definition

**Fraud detection systems often benefit from ensemble learning because fraud behavior is:**

* non-linear
* sparse
* noisy
* highly imbalanced
* behaviorally complex

Single models may miss hidden fraud interactions.

**Ensemble methods combine multiple learners to improve:**

* robustness
* generalization
* fraud recall
* precision stability

**In this notebook, we explore:**

1. Bagging Classifier
2. Extra Trees Classifier
3. Voting Classifier
4. Stacking Classifier

## 2. Mathematical Intuition

**Ensemble Learning**

Ensemble learning combines multiple weak or moderate learners:

$$\hat{y} = \sum_{i=1}^{n} w_i f_i(x)$$

Where:

* f_i(x) = individual model predictions
* w_i = model weights

Goal:

* reduce variance
* reduce overfitting
* improve stability
* capture diverse fraud patterns

---

**Bagging**

Bootstrap Aggregating:

* trains multiple models on random samples
* averages predictions

Benefits:

* lower variance
* more stable trees
* better generalization

---

**Extra Trees**

Extremely Randomized Trees:

* random feature selection
* random split thresholds

Benefits:

* faster training
* lower variance
* stronger regularization


**Voting Classifier**

Combines predictions from different algorithms:

* Logistic Regression
* Random Forest
* Gradient learners
* etc.

Soft Voting

Uses probability averaging:

$$\text{P}(y=1) = \frac{1}{n} \sum \text{P}_i(y=1)$$

---

**Stacking**

Meta-learning architecture:

* Base models generate predictions
* Meta-model learns from those predictions

Architecture:
```text
Base Models → Meta Learner → Final Prediction
```

Very common in enterprise ML competitions and production systems.

## 3. Bias vs Variance

| Model | Bias | Variance |
| :--- | :--- | :--- |
| **Single Tree** | Low | High |
| **Bagging** | Low | Low *(Relative to Single Tree)* |
| **Extra Trees** | Slightly Higher | Much Lower |
| **Stacking** | Low | Low |

## 4. Setup & Import

In [ ]:
# Setup
import sys
sys.path.append("..")

# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Ensemble Models
from sklearn.ensemble import (
    BaggingClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier
)

# Base Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# Evaluation
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# Project Modules
from src.data_loader import load_raw_data
from src.preprocess import preprocess_dataset
from src.config import RANDOM_STATE

# Styling
sns.set_style("whitegrid")

## 5. Load & Preprocess Data

In [ ]:
# Load dataset
df = load_raw_data()

# Preprocess
X_train, X_test, y_train, y_test, preprocessor = (
    preprocess_dataset(df)
)

## 6. Bagging Classifier

**Initialize Model**

In [ ]:
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=8,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

**Train Model**

In [ ]:
bagging_model.fit(X_train, y_train)

**Predictions**

In [ ]:
y_pred_bag = bagging_model.predict(X_test)

y_pred_bag_proba = bagging_model.predict_proba(X_test)[:, 1]

**Evaluation**

In [ ]:
print(classification_report(y_test, y_pred_bag))

**ROC-AUC**

In [ ]:
roc_auc_bag = roc_auc_score(
    y_test,
    y_pred_bag_proba
)

print(f"Bagging ROC-AUC: {roc_auc_bag:.4f}")

## 7. Extra Trees Classifier

**Initialize Model**

In [ ]:
extra_trees_model = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

**Train Model**

In [ ]:
extra_trees_model.fit(X_train, y_train)

**Predictions**

In [ ]:
y_pred_extra = extra_trees_model.predict(X_test)

y_pred_extra_proba = (
    extra_trees_model.predict_proba(X_test)[:, 1]
)

**Evaluation**

In [ ]:
print(classification_report(y_test, y_pred_extra))

**ROC-AUC**

In [ ]:
roc_auc_extra = roc_auc_score(
    y_test,
    y_pred_extra_proba
)

print(f"Extra Trees ROC-AUC: {roc_auc_extra:.4f}")

## 8. Voting Classifier

**Build Base Models**

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

et_model = ExtraTreesClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

**Build Voting Ensemble**

In [ ]:
voting_model = VotingClassifier(
    estimators=[
        ("lr", lr_model),
        ("rf", rf_model),
        ("et", et_model)
    ],
    voting="soft",
    n_jobs=-1
)

**Train Model**

In [ ]:
voting_model.fit(X_train, y_train)

**Predictions**

In [ ]:
y_pred_vote = voting_model.predict(X_test)

y_pred_vote_proba = (
    voting_model.predict_proba(X_test)[:, 1]
)

**Evaluation**

In [ ]:
print(classification_report(y_test, y_pred_vote))

**ROC-AUC**

In [ ]:
roc_auc_vote = roc_auc_score(
    y_test,
    y_pred_vote_proba
)

print(f"Voting ROC-AUC: {roc_auc_vote:.4f}")

## 9. Stacking Classifier

**Build Base Learners**

In [ ]:
base_models = [
    (
        "rf",
        RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    ),
    (
        "et",
        ExtraTreesClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    ),
    (
        "lr",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
]

**Meta Learner**

In [ ]:
meta_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

**Build Stacking Model**

In [ ]:
stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    passthrough=False,
    n_jobs=-1
)

**Train Model**

In [ ]:
stacking_model.fit(X_train, y_train)

**Predictions**

In [ ]:
y_pred_stack = stacking_model.predict(X_test)

y_pred_stack_proba = (
    stacking_model.predict_proba(X_test)[:, 1]
)

**Evaluation**

In [ ]:
print(classification_report(y_test, y_pred_stack))

**ROC-AUC**

In [ ]:
roc_auc_stack = roc_auc_score(
    y_test,
    y_pred_stack_proba
)

print(f"Stacking ROC-AUC: {roc_auc_stack:.4f}")

## 10. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Bagging",
        "Extra Trees",
        "Voting",
        "Stacking"
    ],
    "ROC_AUC": [
        roc_auc_bag,
        roc_auc_extra,
        roc_auc_vote,
        roc_auc_stack
    ]
})

results.sort_values(
    by="ROC_AUC",
    ascending=False
)

## 11. ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(
    y_test,
    y_pred_bag_proba,
    name="Bagging",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test,
    y_pred_extra_proba,
    name="Extra Trees",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test,
    y_pred_vote_proba,
    name="Voting",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test,
    y_pred_stack_proba,
    name="Stacking",
    ax=ax
)

plt.title("Ensemble Models ROC Comparison")
plt.show()

## 12. Feature Importance (Extra Trees)

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": extra_trees_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

feature_importance.head(15)

**Visualization**

In [ ]:
plt.figure(figsize=(10, 8))

sns.barplot(
    data=feature_importance.head(15),
    x="importance",
    y="feature"
)

plt.title("Extra Trees Feature Importance")

plt.show()

## 13. Interpretation

**findings:**

* Stacking produce the best overall ROC-AUC
* Voting improves robustness
* Extra Trees handles noisy fraud features effectively
* Ensemble averaging reduces overfitting

**Important features remain dominant:**

* anomaly_score
* device_anomaly_interaction
* composite_risk_score
* authentication_risk_score

---

## 14. Business Insights

**Ensemble systems are powerful for banking fraud because they:**

* aggregate multiple fraud perspectives
* improve stability across changing fraud patterns
* reduce false positives
* improve detection consistency

**Operational Impact**

**Better ensemble calibration means:**

* fewer missed fraud cases
* better investigator prioritization
* reduced customer friction
* improved financial protection

---

## 15. Limitations

**Ensemble systems:**

* are harder to interpret
* require more compute
* increase inference latency
* complicate deployment pipelines

Stacking especially can become difficult to maintain in production.

---

## 16. Conclusion

**In this notebook we:**

* implemented multiple ensemble architectures
* compared bagging vs stacking approaches
* evaluated fraud detection performance
* analyzed ensemble robustness
* studied feature importance behavior

Ensemble learning significantly improves fraud detection stability and will serve as a strong foundation for boosting methods in the next notebook.